In [ ]:
import torch
import numpy as np
from torch import nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
import os
from sklearn.model_selection import train_test_split
from torch import optim


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Assuming your dataset is in a folder named 'PlantVillage' in your Google Drive
!cp -r drive/MyDrive/PlantVillage /content/
path="/content/PlantVillage"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
full_dataset = datasets.ImageFolder(root=path, transform=None)
class_to_idx = full_dataset.class_to_idx
idx_to_class = {value: key for key, value in class_to_idx.items()}
classes = list(class_to_idx.keys())
num_classes = len(classes)
print(f"Number of classes: {num_classes}")

targets = [sample[1] for sample in full_dataset.samples]

train_idx, test_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    random_state=42,
    stratify=targets
)
print(f"Number of Train images: {len(train_idx)} | Number of Test images: {len(test_idx)}")

Number of classes: 15
Number of Train images: 16510 | Number of Test images: 4128


In [ ]:
def get_mean_std(dataset, batch_size=16, num_workers=2):
  loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)
  # Initialize for single channel (grayscale)
  sum_pixels = 0.0
  sumsq_pixels = 0.0
  count_pixels = 0

  for x, _ in loader:
    b, c, h, w = x.shape
    count_pixels += b * h * w
    # Sum and sumsq for single channel (grayscale)
    sum_pixels += x.sum().item() # Sum over all dimensions
    sumsq_pixels += (x**2).sum().item() # Sum of squares over all dimensions


  mean = sum_pixels / count_pixels
  std = torch.sqrt(torch.tensor(sumsq_pixels / count_pixels - mean ** 2)) # Use torch.tensor for calculation
  return mean, std


# Create a temporary dataset with grayscale transformation to calculate mean and std
temp_full_gray = datasets.ImageFolder(root=path, transform=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
]))

# Calculate mean and std for grayscale images using the full temporary dataset
mean_gray, std_gray = get_mean_std(temp_full_gray)


print(f"Mean for grayscale images: {mean_gray}")
print(f"Standard deviation for grayscale images: {std_gray}")

Mean for grayscale images: 0.4632016388114303
Standard deviation for grayscale images: 0.16794970631599426


In [ ]:
# data transformation for grey scale image
train_transform_gray = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean_gray], std=[std_gray])
])

test_transform_gray = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean_gray], std=[std_gray])
])

In [ ]:
# Load the dataset with grayscale transformations
train_data_gray_all = datasets.ImageFolder(root=path, transform=train_transform_gray)
test_data_gray_all = datasets.ImageFolder(root=path, transform=test_transform_gray)

# Create subsets for training and testing
train_set_gray = Subset(train_data_gray_all, train_idx)
test_set_gray = Subset(test_data_gray_all, test_idx)

# Create DataLoaders
train_loader_gray = DataLoader(train_set_gray, batch_size=32, shuffle=True, num_workers=0)
test_loader_gray = DataLoader(test_set_gray, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
class newCNN_gray(nn.Module):   # model
  def __init__(self, num_classes=15):
    super().__init__()

    self.block1 = nn.Sequential(
        nn.Conv2d(in_channels=1,out_channels=32, kernel_size=3, padding=1), # Changed in_channels to 1
        nn.ReLU(),
        nn.Conv2d(in_channels=32,out_channels=32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

    self.block2 = nn.Sequential(
        nn.Conv2d(in_channels=32,out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=64,out_channels=64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)

    )

    self.block3 = nn.Sequential(
        nn.Conv2d(in_channels=64,out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=128,out_channels=128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=128*28*28, out_features=512), # Corrected in_features here
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(in_features=512, out_features=num_classes),


    )

  def forward(self, x):

    x = self.block1(x)

    x = self.block2(x)

    x = self.block3(x)

    x = self.classifier(x)
    return x

In [ ]:
# Create an instance of newCNN_gray
model_gray = newCNN_gray(num_classes=num_classes).to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
# Changed optimizer to AdamW and initial learning rate
optimizer = optim.AdamW(model_gray.parameters(), lr=0.0005)

# Training loop
epochs = 60 # Increased epochs
best_accuracy = 0.0
for epoch in range(epochs):
    model_gray.train() # Set model to training mode
    running_loss = 0.0
    for inputs, labels in train_loader_gray:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model_gray(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_set_gray)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

    # Evaluation loop after each epoch
    model_gray.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader_gray:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_gray(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the {total} test images: {accuracy:.2f}%')

    # Save the best model
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        torch.save(model_gray.state_dict(), 'best_newCNN_gray_model.pth')
        print(f"Saved best model with accuracy: {best_accuracy:.2f}%")

print(f'\nFinal Best Accuracy: {best_accuracy:.2f}%')

# Save the final trained model's state dictionary to Google Drive
torch.save(model_gray.state_dict(), '/content/drive/MyDrive/final_newCNN_gray_model.pth')
print("\nFinal model state dictionary saved to '/content/drive/MyDrive/final_newCNN_gray_model.pth'")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x100352 and 32768x512)

In [ ]:
loaded_model_gray = newCNN_gray(num_classes=num_classes)

# Then load the saved state dictionary into the model
# Load the model from the path where the final trained model was saved,
# mapping to CPU if necessary
loaded_model_gray.load_state_dict(torch.load('/content/drive/MyDrive/final_newCNN_gray_model.pth', map_location=torch.device('cpu')))

# Move the loaded model to the appropriate device (which is CPU in this case)
loaded_model_gray.to(device)

# Set the loaded model to evaluation mode for inference
loaded_model_gray.eval()

print("Model loaded successfully!")

In [ ]:
image_file_path = '/content/PlantVillage/Potato___Early_blight/009c8c31-f22d-4ffd-8f16-189c6f06c577___RS_Early.B 7885.JPG'

# Load the image using PIL
from PIL import Image
# Ensure the image is in RGB format before converting to grayscale
sample_image = Image.open(image_file_path).convert('RGB')


input_tensor = test_transform_gray(sample_image)
input_batch = input_tensor.unsqueeze(0) # Add a batch dimension

# Move the input to the same device as the model
input_batch = input_batch.to(device)

# Ensure the loaded model is available and in evaluation mode
# If you ran the cell to load the model, loaded_model_gray should be available
if 'loaded_model_gray' not in locals():
    print("Loading the model first...")
    loaded_model_gray = newCNN_gray(num_classes=num_classes)

    # Load the saved state dictionary into the model
    loaded_model_gray.load_state_dict(torch.load('/content/drive/MyDrive/final_newCNN_gray_model.pth')) # Use the final saved model
    # Move the loaded model to the appropriate device
    loaded_model_gray.to(device)
    # Set the loaded model to evaluation mode for inference
    loaded_model_gray.eval()
    print("Model loaded.")


# Perform inference using the loaded model
with torch.no_grad():
    output = loaded_model_gray(input_batch)

# Apply Softmax to get probabilities
probabilities = nn.Softmax(dim=1)(output)

# Get the predicted class index using argmax
predicted_idx = torch.argmax(probabilities, 1)

predicted_idx = predicted_idx.item()

# Get the class name using idx_to_class dictionary
predicted_class_name = idx_to_class[predicted_idx]

print(f"The predicted class for the image at is: {predicted_class_name}")
print(f"Class probabilities: {probabilities}")

**Training Text Model**


In [ ]:
import requests
import pandas as pd

# Define the path for the dataset
parquet_file_path = "hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet"

# Load the data from the parquet file
try:
    df = pd.read_parquet(parquet_file_path)

    # Display the first few rows of the DataFrame
    display(df.head())

except Exception as e:
    print(f"Error: Unable to load data from parquet file. {e}")
    df = pd.DataFrame() # Create an empty DataFrame in case of an error

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [ ]:
# Drop the 'image' column
df_processed = df.drop(columns=['image'])

# Explode the 'captions' column
if 'captions' in df_processed.columns:
    df_processed = df_processed.explode('captions')

# Drop duplicate rows after exploding captions
df_processed = df_processed.drop_duplicates()

    # You might want to normalize the 'objects' column if it contains dictionaries
    # For example, to create columns for keys within the dictionaries:
    # df_processed = pd.concat([df_processed.drop('objects', axis=1), df_processed['objects'].apply(pd.Series)], axis=1)


# Display the first few rows of the processed DataFrame
display(df_processed.head())

,caption,captions
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...
0,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
0,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
0,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
1,Tomato Late blight,A tomato leaf showing dark brown lesions and w...


In [ ]:
df['caption'].unique().tolist()

['Tomato healthy',
 'Tomato Late blight',
 'Tomato mosaic virus',
 'Pepper bell healthy',
 'Potato Early blight',
 'Tomato Early blight',
 'Tomato YellowLeaf Curl Virus',
 'Tomato Target Spot',
 'Pepper bell Bacterial spot',
 'Tomato Septoria leaf spot',
 'Tomato Spider mites Two spotted spider mite',
 'Tomato Bacterial spot',
 'Potato Late blight',
 'Tomato Leaf Mold',
 'Potato healthy']

In [ ]:
import os
import shutil

# Define the source and destination paths
csv_file_path = 'plantvillage_data.csv'
drive_path = '/content/drive/MyDrive/plantvillage_data.csv' # You can change the path in your Drive

# Ensure the source file exists
if os.path.exists(csv_file_path):
    # Copy the file to Google Drive
    shutil.copyfile(csv_file_path, drive_path)
    print(f"'{csv_file_path}' successfully saved to '{drive_path}'")
else:
    print(f"Error: '{csv_file_path}' not found.")

Error: 'plantvillage_data.csv' not found.


In [ ]:
# Define the output CSV file path
csv_file_path = 'plantvillage_data.csv'

# Save the processed DataFrame to a CSV file
df_processed.to_csv(csv_file_path, index=False)

print(f"Processed data saved to {csv_file_path}")

Processed data saved to plantvillage_data.csv


Data File is saved in drive

In [ ]:
import pandas as pd

# Define the path to the CSV file in your Google Drive
drive_path = '/content/drive/MyDrive/plantvillage_data.csv'

# Load the CSV file into a pandas DataFrame
try:
    df_loaded = pd.read_csv(drive_path)

    # Display the first few rows of the loaded DataFrame
    display(df_loaded.head())

except FileNotFoundError:
    print(f"Error: The file '{drive_path}' was not found.")
except Exception as e:
    print(f"Error: An error occurred while loading the file. {e}")

,caption,captions
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d..."
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli..."
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect..."
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [ ]:
encoder = LabelEncoder()
encoder.fit(df_loaded["caption"].tolist())
df_loaded["label"] = encoder.transform(df_loaded["caption"].tolist())
display(df_loaded.head())

,caption,captions,label
0,Tomato healthy,A vibrant green and healthy tomato leaf with s...,13
1,Tomato healthy,"A healthy Solanum lycopersicum leaf, free of d...",13
2,Tomato healthy,"A fresh tomato leaf outdoors, glowing in sunli...",13
3,Tomato healthy,"A clean and healthy tomato leaf image, perfect...",13
4,Tomato Late blight,A tomato leaf showing dark brown lesions and w...,7


In [ ]:
df_train, df_test = train_test_split(df_loaded, train_size=0.8)

In [ ]:
from datasets import Dataset

In [ ]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [ ]:
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def mapper_function(data):
    return tokenizer(data['captions'], truncation=True)

tokenized_train = train_dataset.map(mapper_function, batched=True)
tokenized_test = test_dataset.map(mapper_function, batched=True)

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

In [ ]:
model_name = "distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels= 15)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate
import torch
from transformers import (
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer)
acc_metric = evaluate.load("accuracy")

def compute_metrics(pred):
    logits = pred.predictions
    labels = pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"]}


In [ ]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir="checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5, # Further reduced learning rate
    num_train_epochs=10, # Increased epochs
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    label_smoothing_factor=0.1,
    max_grad_norm=1.0,
)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-4,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
    )

In [ ]:
trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test,
    tokenizer= tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
    )

/tmp/ipython-input-3613370672.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss
4128,0.044200
8256,0.048900
12384,0.000500
16512,0.000500


TrainOutput(global_step=16512, training_loss=0.023525978878949038, metrics={'train_runtime': 949.0027, 'train_samples_per_second': 278.36, 'train_steps_per_second': 17.399, 'total_flos': 1929710488230630.0, 'train_loss': 0.023525978878949038, 'epoch': 4.0})

In [ ]:
# Save the trained model
trainer.save_model("/content/drive/MyDrive/distilbert_plantvillage_text_model")
print("Model saved to Google Drive.")

Model saved to Google Drive.


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Define the path where you saved your model in Google Drive
model_path = "/content/drive/MyDrive/distilbert_plantvillage_text_model"

# Load the tokenizer and model
try:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    loaded_text_model = AutoModelForSequenceClassification.from_pretrained(model_path)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please make sure the model path is correct and the model was saved successfully.")

# You can now use 'loaded_text_model' for inference

Model loaded successfully!


In [ ]:
import torch
from torch.nn.functional import softmax

def predict_text_class_with_argmax(text, model, tokenizer, label_encoder):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # Move inputs to the appropriate device if necessary (e.g., GPU if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}


    # Get model predictions
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        outputs = model(**inputs)

    # Apply softmax to get probabilities
    probabilities = softmax(outputs.logits, dim=1)

    # Get the predicted class index using argmax
    predicted_idx = torch.argmax(probabilities, dim=1).item()

    # Decode the predicted index back to the class name
    predicted_class = label_encoder.inverse_transform([predicted_idx])[0]

    return predicted_class, probabilities

# Assuming 'encoder' is your LabelEncoder instance from earlier
# If not, make sure you have fitted it on the original 'caption' column

# Example usage:
sample_text = "A tomato leaf with brown spots"
predicted_class, probabilities = predict_text_class_with_argmax(sample_text, loaded_text_model, tokenizer, encoder)

print(f"The predicted class for the text '{sample_text}' is: {predicted_class}")
print(f"Class probabilities: {probabilities}")

The predicted class for the text 'A tomato leaf with brown spots' is: Tomato mosaic virus
Class probabilities: tensor([[2.6082e-07, 2.1890e-08, 7.0541e-07, 6.7483e-07, 8.7812e-08, 1.0046e-07,
         1.2443e-06, 4.6533e-07, 3.7000e-10, 2.6596e-10, 1.4317e-09, 1.0021e-06,
         1.8373e-09, 2.5665e-08, 1.0000e+00]], device='cuda:0')


In [ ]:


# Get the true labels from the test dataset
true_labels = tokenized_test["label"]

# Make predictions on the test dataset
# You might want to use a DataLoader for the test set for batch processing

num_samples_to_predict = 5
sample_texts = tokenized_test["captions"][:num_samples_to_predict]

predicted_classes = []
for text in sample_texts:
    predicted_class, _ = predict_text_class_with_argmax(text, loaded_text_model, tokenizer, encoder)
    predicted_classes.append(predicted_class)

# Get the true class names for the sample texts
true_class_names = [encoder.inverse_transform([label])[0] for label in true_labels[:num_samples_to_predict]]

# Print the true and predicted labels for the sample texts
print("Comparing True and Predicted Labels for Sample Texts:")
for i in range(num_samples_to_predict):
    print(f"Text: {sample_texts[i]}")
    print(f"True Label: {true_class_names[i]}")
    print(f"Predicted Label: {predicted_classes[i]}")
    print("-" * 20)

Comparing True and Predicted Labels for Sample Texts:
Text: A tomato leaf showing yellow spots on the upper surface and moldy growth on the underside.
True Label: Tomato Leaf Mold
Predicted Label: Tomato Leaf Mold
--------------------
Text: A labeled image of a tomato leaf showing leaf mold symptoms for agricultural datasets.
True Label: Tomato Leaf Mold
Predicted Label: Tomato Leaf Mold
--------------------
Text: A labeled image of a tomato leaf affected by target spot disease for agricultural datasets.
True Label: Tomato Target Spot
Predicted Label: Tomato Target Spot
--------------------
Text: A tomato plant infected with Passalora fulva, featuring mold growth and chlorotic lesions.
True Label: Tomato Leaf Mold
Predicted Label: Tomato Leaf Mold
--------------------
Text: A tomato leaf showing dark brown lesions and water-soaked edges, symptoms of late blight disease.
True Label: Tomato Late blight
Predicted Label: Tomato Late blight
--------------------
